# Neural Network from Scratch (NumPy)
This notebook builds a simple neural network for handwritten digit classification using only NumPy.

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt

# Import required libraries.

In [ ]:
from google.colab import drive

# Mount Google Drive so the dataset can be accessed.
drive.mount('/content/drive')

In [ ]:
# Read the training dataset.
data = pd.read_csv('/content/drive/MyDrive/digit-recognizer/train.csv')

# Display the first five rows.
data.head()

In [ ]:
# Convert DataFrame to NumPy array for faster computation.
data = np.array(data)

# m = number of examples, n = number of columns.
m, n = data.shape

# Shuffle the dataset before splitting.
np.random.shuffle(data)

print(m, n)

In [ ]:
# Split into 80% training and 20% validation.
train_data = data[0:int(0.8*m), :]
val_data = data[int(0.8*m):m, :]

# First column contains labels.
Y_train = train_data[:, 0]
Y_val = val_data[:, 0]

# Remaining columns are pixel values.
X_train = train_data[:, 1:].T / 255.0
X_val = val_data[:, 1:].T / 255.0

print(X_train.shape)
print(Y_train.shape)
print(X_val.shape)
print(Y_val.shape)

In [ ]:
def initialize_parameters():
    # Randomly initialize weights and biases.
    W1 = np.random.rand(10, 784) - 0.5
    B1 = np.random.rand(10, 1) - 0.5
    W2 = np.random.rand(10, 10) - 0.5
    B2 = np.random.rand(10, 1) - 0.5
    return W1, B1, W2, B2

def ReLU(X):
    # ReLU activation keeps positive values and replaces negatives with 0.
    return np.maximum(X, 0)

def softmax_calculator(Z):
    # Convert scores into probabilities.
    return np.exp(Z) / np.sum(np.exp(Z), axis=0)

def forward_propagation(W1, B1, W2, B2, X):
    # Hidden layer.
    Z1 = W1.dot(X) + B1
    A1 = ReLU(Z1)

    # Output layer.
    Z2 = W2.dot(A1) + B2
    A2 = softmax_calculator(Z2)

    return Z1, A1, Z2, A2

def one_hot_converter(Y):
    # Convert labels into one-hot vectors.
    one_hot_Y = np.zeros((Y.size, Y.max() + 1))
    one_hot_Y[np.arange(Y.size), Y] = 1
    return one_hot_Y.T

def backward_propagation(W1, B1, W2, B2, Z1, A1, Z2, A2, X, Y):
    # Compute gradients using backpropagation.
    one_hot_Y = one_hot_converter(Y)

    dZ2 = A2 - one_hot_Y
    dW2 = (1 / m) * dZ2.dot(A1.T)
    dB2 = (1 / m) * np.sum(dZ2)

    dZ1 = W2.T.dot(dZ2) * (Z1 > 0)
    dW1 = (1 / m) * dZ1.dot(X.T)
    dB1 = (1 / m) * np.sum(dZ1)

    return dW1, dB1, dW2, dB2

def update_parameters(W1, B1, W2, B2, dW1, dB1, dW2, dB2, learning_rate):
    # Update parameters using gradient descent.
    W1 -= learning_rate * dW1
    B1 -= learning_rate * dB1
    W2 -= learning_rate * dW2
    B2 -= learning_rate * dB2
    return W1, B1, W2, B2

def get_predictions(A2):
    # Choose the class with highest probability.
    return np.argmax(A2, 0)

def get_accuracy(predictions, Y):
    # Calculate classification accuracy.
    return np.sum(predictions == Y) / Y.size

def gradient_descent(X, Y, alpha, iterations):
    W1, B1, W2, B2 = initialize_parameters()

    for i in range(iterations):
        Z1, A1, Z2, A2 = forward_propagation(W1, B1, W2, B2, X)
        dW1, dB1, dW2, dB2 = backward_propagation(W1, B1, W2, B2, Z1, A1, Z2, A2, X, Y)
        W1, B1, W2, B2 = update_parameters(W1, B1, W2, B2, dW1, dB1, dW2, dB2, alpha)

        if i % 20 == 0:
            print("Iteration:", i)
            print("Accuracy:", get_accuracy(get_predictions(A2), Y))

    return W1, B1, W2, B2

In [ ]:
# Train the neural network.
W1, B1, W2, B2 = gradient_descent(X_train, Y_train, 0.1, 1000)

In [ ]:
# Predict one validation image.
val_index = 560

Z1val, A1val, Z2val, A2val = forward_propagation(
    W1, B1, W2, B2, X_val[:, val_index, None]
)

print("Predicted:", get_predictions(A2val))
print("Actual:", Y_val[val_index])

image_array = X_val[:, val_index].reshape(28, 28)
plt.imshow(image_array, cmap="gray")
plt.show()

In [ ]:
# Evaluate validation accuracy.
Z1val, A1val, Z2val, A2val = forward_propagation(W1, B1, W2, B2, X_val)

val_acc = get_accuracy(get_predictions(A2val), Y_val)
print("Validation Accuracy:", val_acc)